# Homework #4

By: Jenny Park\
Date: April 13, 2026\
Class: QMSS GR5069

In [0]:
import pandas as pd
from pyspark.sql import functions as F

In [0]:
!pip install xgboost

## **Prediction Problem:** Whether or not the driver will finish the race

#### Building the dataset

In [0]:
# loading in datasets
df_results = spark.read.csv('/Volumes/gr5069/raw/f1_data/results.csv', header=True)

df_drivers = spark.read.csv('/Volumes/gr5069/raw/f1_data/drivers.csv', header=True)

df_races = spark.read.csv('/Volumes/gr5069/raw/f1_data/races.csv', header=True)

df_constructors = spark.read.csv('/Volumes/gr5069/raw/f1_data/constructors.csv', header=True)

df_status = spark.read.csv('/Volumes/gr5069/raw/f1_data/status.csv', header=True)

In [0]:
display(df_status)

In [0]:
# joining datasets
df = df_results.join(df_drivers, on='driverId', how='inner')
df = df.join(df_races, on='raceId', how='inner')
df = df.join(df_constructors, on='constructorId', how='inner')
df = df.join(df_status, on='statusId', how='inner')

# selecting features
df = df.select('constructorId', 'raceId', 'year', 'round', 'circuitId', 'driverId', 'grid', 'laps', 'statusId')

####Preprocessing steps

In [0]:
# identify categorical columns
df = df.toPandas()

categorical_columns = ['raceId', 'driverId', 'constructorId', 'circuitId', 'grid', 'year', 'round']

for col in categorical_columns:
    df[col] = df[col].astype('category')


# identify numerical columns
numerical_columns = ['laps']

for col in numerical_columns:
    df[col] = df[col].astype('float')


# prepare target column
df['statusId'] = df['statusId'].astype('int')
df.loc[df['statusId'] != 1, 'statusId'] = 0

df = df.select_dtypes(include=['category', 'float', 'int'])

In [0]:
# confirming dataset contents
display(df)
print(df['statusId'].value_counts())

####Perform a train/test split

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(['statusId'], axis=1), df[['statusId']].values.ravel(), random_state=42)

#### Building the model

In [0]:
import mlflow.sklearn

from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
import seaborn as sns
import tempfile
import matplotlib.pyplot as plt


def log_xgb(experimentID, run_name, params, X_train, X_test, y_train, y_test):

    with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:

        # Create model, train it, and create predictions
        model = XGBClassifier(**params)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]
        
        # Log model
        mlflow.sklearn.log_model(model, 'xgboost-model')
        
        # Log parameters
        [mlflow.log_param(param, value) for param, value in params.items()]

        # Create metrics
        accuracy = accuracy_score(y_test, predictions)
        f1 = f1_score(y_test, predictions)
        precision = precision_score(y_test, predictions)
        recall = recall_score(y_test, predictions)
        roc_auc = roc_auc_score(y_test, probs)
        average_precision = average_precision_score(y_test, probs)

        # Print metrics
        print(f'Accuracy: ', accuracy)
        print(f'F1: ', f1)
        print(f'Precision: ', precision)
        print(f'Recall: ', recall)
        print(f'ROC AUC: ', roc_auc)
        print(f'Average precision: ', average_precision)

        # Log metrics
        mlflow.log_metric('accuracy', accuracy)
        mlflow.log_metric('f1', f1)
        mlflow.log_metric('precision', precision)
        mlflow.log_metric('recall', recall)
        mlflow.log_metric('roc_auc', roc_auc)
        mlflow.log_metric('average_precision', average_precision)

        # Create feature importance
        importance = pd.DataFrame(list(zip(X_train.columns, model.feature_importances_)), columns=['Feature', 'Importance']).sort_values('Importance', ascending=False)

        # Log importances using a temporary file
        temp = tempfile.NamedTemporaryFile(prefix='feature-importance-', suffix='.csv') 
        temp_name = temp.name
        try:
            importance.to_csv(temp_name, index=False)
            mlflow.log_artifact(temp_name, artifact_path='feature-importance.csv')
        finally:
            temp.close() # Delete the temp file


        # Plot feature importance
        plt.figure(figsize=(10, 6))
        sns.barplot(data=importance.head(5), x='Importance', y='Feature')
        plt.title('Top 5 Feature Importances')
        plt.tight_layout()
        plt.show()

        # Log plots using a temporary file
        plot_temp = tempfile.NamedTemporaryFile(prefix='feature-importance-', suffix='.png', delete=False)
        plot_path = plot_temp.name
        try:
            plt.savefig(plot_path)
            mlflow.log_artifact(plot_path, artifact_path='feature-importance')
        finally:
            plt.close()
            plot_temp.close()

        
        return run.info.run_id, f1

## Experiments

In [0]:
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

param_grid = [
    # experiment 1: baseline
    {
        'objective': 'binary:logistic', 
        'max_depth': 5, 
        'learning_rate': 0.05, 
        'n_estimators': 300, 
        'subsample': 0.8, 
        'colsample_bytree': 0.8, 
        'scale_pos_weight': scale_pos_weight, 
        'enable_categorical': True
    },

    # experiment 2: deeper tree
    {
        'objective': 'binary:logistic', 
        'max_depth': 8, 
        'learning_rate': 0.05, 
        'n_estimators': 300, 
        'subsample': 0.8, 
        'colsample_bytree': 0.8, 
        'scale_pos_weight': scale_pos_weight, 
        'enable_categorical': True
    },

    # experiment 3: shallower tree
    {
        'objective': 'binary:logistic', 
        'max_depth': 3, 
        'learning_rate': 0.05, 
        'n_estimators': 300, 
        'subsample': 0.8, 
        'colsample_bytree': 0.8, 
        'scale_pos_weight': scale_pos_weight, 
        'enable_categorical': True
    },

    # experiment 4: lower learning rate
    {
        'objective': 'binary:logistic', 
        'max_depth': 5, 
        'learning_rate': 0.01, 
        'n_estimators': 300, 
        'subsample': 0.8, 
        'colsample_bytree': 0.8, 
        'scale_pos_weight': scale_pos_weight, 
        'enable_categorical': True
    },

    # experiment 5: higher learning rate
    {
        'objective': 'binary:logistic', 
        'max_depth': 5, 
        'learning_rate': 0.5, 
        'n_estimators': 300, 
        'subsample': 0.8, 
        'colsample_bytree': 0.8, 
        'scale_pos_weight': scale_pos_weight, 
        'enable_categorical': True
    },

    # experiment 6: high n_estimators
    {
        'objective': 'binary:logistic', 
        'max_depth': 5, 
        'learning_rate': 0.05, 
        'n_estimators': 600, 
        'subsample': 0.8, 
        'colsample_bytree': 0.8, 
        'scale_pos_weight': scale_pos_weight, 
        'enable_categorical': True
    },

    # experiment 7: low n_estimators
    {
        'objective': 'binary:logistic', 
        'max_depth': 5, 
        'learning_rate': 0.05, 
        'n_estimators': 150, 
        'subsample': 0.8, 
        'colsample_bytree': 0.8, 
        'scale_pos_weight': scale_pos_weight, 
        'enable_categorical': True
    },

# experiment 8: shallow tree + higher learning rate
    {
        'objective': 'binary:logistic', 
        'max_depth': 3, 
        'learning_rate': 0.15, 
        'n_estimators': 600, 
        'subsample': 0.8,  
        'colsample_bytree': 0.8, 
        'scale_pos_weight': scale_pos_weight, 
        'enable_categorical': True
    },

    # experiment 9: deeper tree + lower learning rate
    {
        'objective': 'binary:logistic', 
        'max_depth': 10, 
        'learning_rate': 0.01, 
        'n_estimators': 400, 
        'subsample': 0.7, 
        'colsample_bytree': 0.7, 
        'scale_pos_weight': scale_pos_weight, 
        'enable_categorical': True
    },

    # experiment 10: deeper tree + higher learning rate + high n_estimators
    {
        'objective': 'binary:logistic', 
        'max_depth': 10, 
        'learning_rate': 0.15, 
        'n_estimators': 600, 
        'subsample': 0.9, 
        'colsample_bytree': 0.9, 
        'scale_pos_weight': scale_pos_weight, 
        'enable_categorical': True
    }
]

experiment = mlflow.set_experiment('/Users/jjp2219@columbia.edu/F1_XGBoost')
experiment_ID = experiment.experiment_id

f1_scores = []
run_names = []

for i, params in enumerate(param_grid):
    
    run_name = f'F1 XGBoost Parameter Tuning: Run {i+1}'
    
    print(f"Running experiment {i+1}/10: {run_name}")
    
    run_id, f1 = log_xgb(experiment_ID, run_name, params, X_train, X_test, y_train, y_test)

    f1_scores.append(f1)
    run_names.append(run_name)

    print()
    print()


## Best Run Analysis

In [0]:
# Create plot
plt.plot(run_names, f1_scores)

# Add titles and labels
plt.title('F1 Scores of Experiments')
plt.xlabel('Run Name')
plt.ylabel('F1-Scores')
plt.xticks(rotation=45, fontsize=8) # 45 degrees, size 8

# Show plot
plt.show()

Based on the F1 scores of each experiment, Run #10 performed the best. This model combined a deeper tree with a higher learning rate and a high number of estimators, which produced an F1 score of almost 91%. 

I chose to use F1 as the primary evaluation metric due to the unbalanced nature of the classification problem. DNFs occur much more frequently than those who finish the race. Inherently, the F1 score captures best the balance between recall and precision, and enables a balances evaluation of how each model is performing. 